In [2]:
!pip install youtube-transcript-api yt-dlp -q

In [3]:
# Cell 2 — YouTube transcript functions
import re
import subprocess
import tempfile
import os


def extract_video_id(url):
    """Extract YouTube video ID from any URL format."""
    patterns = [
        r'(?:v=)([^&\n?#]+)',
        r'(?:youtu\.be/)([^&\n?#]+)',
        r'(?:shorts/)([^&\n?#]+)',
        r'(?:embed/)([^&\n?#]+)',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError(f"Cannot extract video ID from: {url}")


def get_youtube_transcript(video_url):
    """Fetch transcript using youtube-transcript-api, falls back to yt-dlp."""
    video_id = extract_video_id(video_url)
    print(f"📹 Video ID: {video_id}")

    # ── Method 1: youtube-transcript-api (fastest, no API key needed) ──
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        print("🔄 Trying youtube-transcript-api...")

        ytt        = YouTubeTranscriptApi()
        transcript = ytt.fetch(video_id)
        segments   = [
            {
                "start":    round(s.start, 2),
                "duration": round(s.duration, 2),
                "text":     s.text
            }
            for s in transcript.snippets
        ]
        full_text = " ".join(s.text for s in transcript.snippets)

        print(f"✅ Success! Language: {transcript.language} | "
              f"Auto-generated: {transcript.is_generated} | "
              f"Segments: {len(segments)}")

        return {
            "video_id":  video_id,
            "method":    "youtube-transcript-api",
            "language":  transcript.language,
            "generated": transcript.is_generated,
            "segments":  segments,
            "full_text": full_text,
        }

    except Exception as e1:
        print(f"⚠️  Method 1 failed: {e1}")

    # ── Method 2: yt-dlp auto-captions fallback ──────────────────
    try:
        print("🔄 Trying yt-dlp auto-captions...")
        with tempfile.TemporaryDirectory() as tmp:
            subprocess.run([
                "yt-dlp",
                "--write-auto-subs",
                "--skip-download",
                "--sub-format", "vtt",
                "--sub-langs",  "en",
                "-o", f"{tmp}/%(id)s",
                f"https://www.youtube.com/watch?v={video_id}"
            ], capture_output=True, timeout=60)

            vtt_files = [f for f in os.listdir(tmp) if f.endswith(".vtt")]
            if not vtt_files:
                raise FileNotFoundError("No VTT subtitle file found")

            with open(os.path.join(tmp, vtt_files[0]), "r") as f:
                raw = f.read()

            blocks = re.findall(
                r'(\d{2}:\d{2}:\d{2}\.\d{3}) --> (\d{2}:\d{2}:\d{2}\.\d{3})\n(.+?)(?=\n\n|\Z)',
                raw, re.DOTALL
            )
            segments  = []
            seen_text = set()
            for start, end, text in blocks:
                clean = re.sub(r'<[^>]+>', '', text).strip()
                if clean and clean not in seen_text:
                    segments.append({"start": start, "end": end, "text": clean})
                    seen_text.add(clean)

            full_text = " ".join(s["text"] for s in segments)
            print(f"✅ Success via yt-dlp! Segments: {len(segments)}")

            return {
                "video_id":  video_id,
                "method":    "yt-dlp-autocaptions",
                "language":  "en",
                "generated": True,
                "segments":  segments,
                "full_text": full_text,
            }

    except Exception as e2:
        print(f"⚠️  Method 2 failed: {e2}")

    return {"video_id": video_id, "error": "No transcript available for this video"}


def print_yt_transcript(result):
    """Pretty print YouTube transcript result."""
    if "error" in result:
        print(f"\n❌ {result['error']}")
        return

    print(f"\n{'═'*55}")
    print(f"  📝 YOUTUBE TRANSCRIPT")
    print(f"{'═'*55}")
    print(f"  Method    : {result['method']}")
    print(f"  Language  : {result['language']}")
    print(f"  Generated : {result['generated']}")
    print(f"  Segments  : {len(result['segments'])}")
    print(f"\n  📄 Full Text:")
    print(f"  {'─'*50}")
    words, line = result['full_text'].split(), "  "
    for word in words:
        if len(line) + len(word) > 80:
            print(line)
            line = "  " + word + " "
        else:
            line += word + " "
    if line.strip():
        print(line)

    print(f"\n  🕐 Timestamped Segments (first 5):")
    print(f"  {'─'*50}")
    for seg in result['segments'][:5]:
        print(f"  [{seg['start']}s]  {seg['text']}")
    if len(result['segments']) > 5:
        print(f"  ... +{len(result['segments'])-5} more segments")
    print(f"{'═'*55}")

print("✅ YouTube functions loaded successfully")

✅ YouTube functions loaded successfully


In [4]:
# Cell 3 — Run YouTube transcript
url    = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # ← paste any YouTube URL
result = get_youtube_transcript(url)
print_yt_transcript(result)

# Uncomment to print full text only:
# print(result['full_text'])

# Uncomment to print all segments:
# for seg in result['segments']:
#     print(f"[{seg['start']}s]  {seg['text']}")

📹 Video ID: dQw4w9WgXcQ
🔄 Trying youtube-transcript-api...
✅ Success! Language: English | Auto-generated: False | Segments: 61

═══════════════════════════════════════════════════════
  📝 YOUTUBE TRANSCRIPT
═══════════════════════════════════════════════════════
  Method    : youtube-transcript-api
  Language  : English
  Generated : False
  Segments  : 61

  📄 Full Text:
  ──────────────────────────────────────────────────
  [♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules and so do I ♪ ♪ A 
  full commitment's what I'm thinking of ♪ ♪ You wouldn't get this from any 
  other guy ♪ ♪ I just wanna tell you how I'm feeling ♪ ♪ Gotta make you 
  understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never 
  gonna run around and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna 
  say goodbye ♪ ♪ Never gonna tell a lie and hurt you ♪ ♪ We've known each other 
  for so long ♪ ♪ Your heart's been aching but you're too shy to say it ♪ ♪ 
  Inside we both know 

Instagram Transcript extractor

In [5]:
!pip install faster-whisper yt-dlp -q

In [8]:
# Cell 5 — Instagram transcript functions
import re
import os
import subprocess
import tempfile


def get_instagram_transcript(post_url, model_size="tiny"):
    """
    Downloads Instagram reel/video audio and transcribes
    using faster-whisper locally (free, no API key, no kernel crash).

    model_size options (trade-off speed vs accuracy):
        'tiny'   → fastest,  ~10 sec,  basic accuracy
        'base'   → fast,     ~20 sec,  good accuracy   ← recommended
        'small'  → medium,   ~45 sec,  better accuracy
        'medium' → slow,     ~2 min,   best accuracy
    """
    from faster_whisper import WhisperModel

    print(f"📥 Downloading audio from: {post_url}")

    with tempfile.TemporaryDirectory() as tmp:
        audio_path = os.path.join(tmp, "audio.mp3")

        # ── Step 1: Download audio only (lighter than full video) ──
        cmd = [
            "yt-dlp",
            "--no-check-certificates",
            "-x",                        # extract audio only
            "--audio-format", "mp3",
            "--audio-quality", "5",       # medium quality — good enough for speech
            "-o", audio_path,
            post_url
        ]
        dl = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

        # yt-dlp may append extension differently
        if not os.path.exists(audio_path):
            # search for any audio file in tmp
            audio_files = [
                f for f in os.listdir(tmp)
                if f.startswith("audio") and not f.endswith(".part")
            ]
            if not audio_files:
                print(f"⚠️  Download error: {dl.stderr[:300]}")
                return {"error": "Audio download failed. Make sure the reel is public."}
            audio_path = os.path.join(tmp, audio_files[0])

        size_mb = os.path.getsize(audio_path) / (1024 * 1024)
        print(f"✅ Audio downloaded ({size_mb:.1f} MB)")

        # ── Step 2: Transcribe with faster-whisper ─────────────────
        # device='cpu', compute_type='int8' = minimum memory usage
        print(f"🔊 Transcribing with faster-whisper ({model_size})...")
        model = WhisperModel(model_size, device="cpu", compute_type="int8")
        segments_gen, info = model.transcribe(
            audio_path,
            beam_size=1,              # faster inference
            vad_filter=True,          # skip silent parts
            vad_parameters=dict(min_silence_duration_ms=500)
        )

        segments  = []
        full_text = []

        for seg in segments_gen:      # generator — memory efficient
            text = seg.text.strip()
            if text:
                segments.append({
                    "start": round(seg.start, 2),
                    "end":   round(seg.end,   2),
                    "text":  text
                })
                full_text.append(text)

        print(f"✅ Done! Language: {info.language} | "
              f"Confidence: {round(info.language_probability * 100)}% | "
              f"Segments: {len(segments)}")

        return {
            "url":         post_url,
            "method":      f"faster-whisper-{model_size}",
            "language":    info.language,
            "confidence":  f"{round(info.language_probability * 100)}%",
            "segments":    segments,
            "full_text":   " ".join(full_text),
        }


def print_ig_transcript(result):
    """Pretty print Instagram transcript result."""
    if "error" in result:
        print(f"\n❌ {result['error']}")
        return

    print(f"\n{'═'*55}")
    print(f"  📝 INSTAGRAM TRANSCRIPT")
    print(f"{'═'*55}")
    print(f"  Method     : {result['method']}")
    print(f"  Language   : {result['language']}")
    print(f"  Confidence : {result['confidence']}")
    print(f"  Segments   : {len(result['segments'])}")

    print(f"\n  📄 Full Text:")
    print(f"  {'─'*50}")
    if result['full_text']:
        words, line = result['full_text'].split(), "  "
        for word in words:
            if len(line) + len(word) > 80:
                print(line)
                line = "  " + word + " "
            else:
                line += word + " "
        if line.strip():
            print(line)
    else:
        print("  (no speech detected)")

    print(f"\n  🕐 Timestamped Segments:")
    print(f"  {'─'*50}")
    for seg in result['segments']:
        print(f"  [{seg['start']}s → {seg['end']}s]  {seg['text']}")
    print(f"{'═'*55}")

print("✅ Instagram functions loaded successfully")

✅ Instagram functions loaded successfully


In [10]:
# Cell 6 — Run Instagram transcript
url    = "https://www.instagram.com/reel/DXq5z2TyaAF/?igsh=b3RvZ3h5OW03dzAw"  # ← paste reel URL
result = get_instagram_transcript(url, model_size="tiny")  # change to 'base' for better accuracy
print_ig_transcript(result)

# Uncomment to print full text only:
# print(result['full_text'])

# Uncomment to print all segments:
# for seg in result['segments']:
#     print(f"[{seg['start']}s → {seg['end']}s]  {seg['text']}")

📥 Downloading audio from: https://www.instagram.com/reel/DXq5z2TyaAF/?igsh=b3RvZ3h5OW03dzAw
✅ Audio downloaded (0.4 MB)
🔊 Transcribing with faster-whisper (tiny)...
✅ Done! Language: en | Confidence: 96% | Segments: 18

═══════════════════════════════════════════════════════
  📝 INSTAGRAM TRANSCRIPT
═══════════════════════════════════════════════════════
  Method     : faster-whisper-tiny
  Language   : en
  Confidence : 96%
  Segments   : 18

  📄 Full Text:
  ──────────────────────────────────────────────────
  And you may succeed and you may fail And I just want to tell you here, fail, 
  please fail Because I didn't learn anything from success just me I only got a 
  little arrogant Success thought me nothing Fail your thought me everything 
  Lost and failure will be your greatest teachers Please go through it, don't 
  detach yourself from heartbreak Don't detach yourself from pain Don't detach 
  yourself from failure Because they will be your greatest teachers Greatest 
  relati